# Piyu AI Fashion Design Generator — Colab

**Pipeline:** RealVisXL → SAM → IDM-VTON

### What was fixed
| Previous error | Fix |
|---|---|
| `ModuleNotFoundError: No module named 'clip'` | Removed `auto1111sdk`; CLIP installed from OpenAI GitHub |
| `auto1111sdk` Python 3.12 incompatibility | Replaced with native `diffusers.StableDiffusionXLPipeline.from_single_file()` |
| `operator torchvision::nms does not exist` | Pinned matching Torch + Torchvision |
| `PIL._typing._Ink` | Pillow pinned to compatible version |
| `ModuleNotFoundError: No module named 'src'` | Explicit IDM-VTON `PYTHONPATH` |
| GPU OOM between stages | `gc.collect()` + `torch.cuda.empty_cache()` between pipelines |

**Start with a fresh runtime:** Runtime → Disconnect and delete runtime → Reconnect

In [ ]:
# ── STEP 1 — INSTALL DEPENDENCIES ────────────────────────────────────────

import sys, subprocess

print("Python:", sys.version.split()[0])

# Core packages — pinned for Colab T4 compatibility
packages = [
    "numpy==1.26.4",
    "Pillow>=9.5.0,<11.0",
    "opencv-python>=4.9.0",
    "scipy>=1.13.0",
    "diffusers>=0.27.2",
    "transformers>=4.37.2",
    "accelerate>=0.30.0",
    "peft>=0.10.0",
    "huggingface_hub>=0.23.0",
    "tokenizers>=0.15.2",
    "safetensors>=0.4.3",
    "einops>=0.7.0",
    "timm>=0.9.16",
    "onnxruntime>=1.16.3",
    "open-clip-torch>=2.20.0",
    "pycocotools",
    "fvcore",
    "cloudpickle",
    "av",
    "tqdm",
    "python-dotenv",
]

print("\nInstalling packages...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", *packages],
    check=True
)

# OpenAI CLIP — must be installed from source (PyPI 'clip' is a different package)
print("\nInstalling OpenAI CLIP from source...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet",
     "git+https://github.com/openai/CLIP.git"],
    check=True
)

# Segment Anything
print("\nInstalling Segment Anything...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet",
     "git+https://github.com/facebookresearch/segment-anything.git"],
    check=True
)

print("\n✅ STEP 1 complete.")

In [ ]:
# ── STEP 2 — VERIFY IMPORTS ────────────────────────────────────────────────

import sys
import numpy as np
import PIL
import torch
import torchvision
import diffusers
import transformers
import accelerate
import clip  # OpenAI CLIP — should now import cleanly

print("Python:",       sys.version.split()[0])
print("NumPy:",        np.__version__)
print("Pillow:",       PIL.__version__)
print("Torch:",        torch.__version__)
print("Torchvision:",  torchvision.__version__)
print("Diffusers:",    diffusers.__version__)
print("Transformers:", transformers.__version__)
print("Accelerate:",   accelerate.__version__)
print("CLIP:",         clip.__version__ if hasattr(clip, '__version__') else 'installed')

assert torch.cuda.is_available(), "⚠️ GPU is not enabled — enable it in Runtime > Change runtime type"
print("\nGPU:", torch.cuda.get_device_name(0))

# Quick torchvision NMS smoke-test (catches ABI mismatch)
boxes  = torch.tensor([[0.0, 0.0, 100.0, 100.0]])
scores = torch.tensor([0.9])
_ = torchvision.ops.nms(boxes, scores, 0.5)
print("✅ torchvision NMS works.")

from segment_anything import SamPredictor, sam_model_registry
print("✅ SAM import works.")

from diffusers import StableDiffusionXLPipeline
print("✅ StableDiffusionXLPipeline works.")

print("\n🎉 DEPENDENCY CHECK PASSED")

In [ ]:
# ── STEP 3 — CLONE PROJECT + IDM-VTON ─────────────────────────────────────

from pathlib import Path
import subprocess, os, sys

REPO_URL  = "https://github.com/Piyu242005/AI-Fashion-Design-Generator-IBM-INTERSHIP-2026.git"
REPO_ROOT = Path("/content/AI-Fashion-Design-Generator-IBM-INTERSHIP-2026")
PROJECT   = REPO_ROOT / "Piyu-AI-Clothing-Fashion-Design-Generator"
IDM       = PROJECT / "idm_vton"

if not PROJECT.exists():
    if not REPO_ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
    else:
        raise RuntimeError("Repository exists but project folder is missing.")

if not IDM.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/yisol/IDM-VTON.git", str(IDM)],
        check=True
    )

for d in [
    PROJECT / "weights",
    PROJECT / "reference_images",
    PROJECT / "samples",
    PROJECT / "results",
    IDM / "ckpt/densepose",
    IDM / "ckpt/humanparsing",
    IDM / "ckpt/openpose/ckpts",
]:
    d.mkdir(parents=True, exist_ok=True)

os.chdir(PROJECT)

if str(IDM) not in sys.path:
    sys.path.insert(0, str(IDM))

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"]   = "false"
# Point model_manager at the weights directory
os.environ["MODEL_DIR"] = str(PROJECT)

print("PROJECT:",   PROJECT)
print("IDM-VTON:",  IDM)
print("src exists:", (IDM / "src").exists())
print("CWD:",        Path.cwd())

In [ ]:
# ── STEP 4A — FIND OR DOWNLOAD REALVISXL + SAM ────────────────────────────

from pathlib import Path
import os, shutil, subprocess

def find_first(filename):
    for p in Path("/content").rglob(filename):
        if p.is_file():
            return p
    return None

def link_or_copy(src, dst):
    src, dst = Path(src), Path(dst)
    if dst.exists():
        return
    try:
        os.symlink(src.resolve(), dst)
        print("Linked:", dst)
    except OSError:
        shutil.copy2(src, dst)
        print("Copied:", dst)

# ── RealVisXL ──────────────────────────────────────────────────────────────
realvis_dst = PROJECT / "weights/realvisxl.safetensors"
realvis_src = find_first("realvisxl.safetensors")

if realvis_src and realvis_src != realvis_dst:
    print("Found RealVisXL:", realvis_src)
    link_or_copy(realvis_src, realvis_dst)
elif not realvis_dst.exists():
    from huggingface_hub import hf_hub_download
    import os
    HF_TOKEN = os.getenv("HF_TOKEN", "")  # set your token in Colab Secrets
    print("Downloading RealVisXL from Hugging Face...")
    hf_hub_download(
        repo_id="Piyu2420/AI-Fashion-Design-Generator-IBM-INTERSHIP-2026",
        filename="realvisxl/realvisxl.safetensors",
        local_dir=str(PROJECT),
        token=HF_TOKEN or None,
    )
    import shutil
    hf_placed = PROJECT / "realvisxl/realvisxl.safetensors"
    if hf_placed.exists():
        realvis_dst.parent.mkdir(parents=True, exist_ok=True)
        hf_placed.rename(realvis_dst)

# ── SAM ViT-H ─────────────────────────────────────────────────────────────
sam_dst = PROJECT / "weights/sam_vit_h_4b8939.pth"
sam_src = find_first("sam_vit_h_4b8939.pth")

if sam_src and sam_src != sam_dst:
    print("Found SAM:", sam_src)
    link_or_copy(sam_src, sam_dst)
elif not sam_dst.exists():
    print("Downloading SAM ViT-H...")
    subprocess.run([
        "wget", "-c", "-q", "-O", str(sam_dst),
        "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth"
    ], check=True)

print("RealVisXL:", round(realvis_dst.stat().st_size / 1024**3, 2), "GB")
print("SAM:",       round(sam_dst.stat().st_size / 1024**3, 2), "GB")

In [ ]:
# ── STEP 4B — IDM-VTON SUPPORT CHECKPOINTS ────────────────────────────────

from pathlib import Path
import subprocess

downloads = [
    (
        IDM / "ckpt/densepose/model_final_162be9.pkl",
        "https://huggingface.co/spaces/yisol/IDM-VTON/resolve/main/ckpt/densepose/model_final_162be9.pkl?download=true"
    ),
    (
        IDM / "ckpt/humanparsing/parsing_atr.onnx",
        "https://huggingface.co/spaces/yisol/IDM-VTON/resolve/main/ckpt/humanparsing/parsing_atr.onnx?download=true"
    ),
    (
        IDM / "ckpt/humanparsing/parsing_lip.onnx",
        "https://huggingface.co/spaces/yisol/IDM-VTON/resolve/main/ckpt/humanparsing/parsing_lip.onnx?download=true"
    ),
    (
        IDM / "ckpt/openpose/ckpts/body_pose_model.pth",
        "https://huggingface.co/spaces/yisol/IDM-VTON/resolve/main/ckpt/openpose/ckpts/body_pose_model.pth?download=true"
    ),
]

for dst, url in downloads:
    if dst.exists() and dst.stat().st_size > 0:
        print("✅ Already exists:", dst.name)
    else:
        print("Downloading:", dst.name)
        subprocess.run(["wget", "-c", "-q", "-O", str(dst), url], check=True)

print("\n✅ IDM-VTON support checkpoints ready.")

In [ ]:
# ── STEP 5 — SET HF_TOKEN (needed if models are private) ──────────────────
#
# Recommended: use Colab Secrets (key icon in left sidebar)
#   Add a secret named HF_TOKEN with your Hugging Face token.

import os
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
        print("✅ HF_TOKEN loaded from Colab Secrets.")
    else:
        print("⚠️  HF_TOKEN not set in Colab Secrets — public repos only.")
except Exception:
    print("ℹ️  Not in Colab Secrets context — using env var if set.")

print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))

In [ ]:
# ── STEP 6 — GENERATE AI FASHION MODEL ────────────────────────────────────
#
# Uses generate_model.py (updated to use diffusers — no auto1111sdk, no clip)

import subprocess, sys
from pathlib import Path

MODEL_IMAGE = PROJECT / "reference_images/fashion_model.png"

PROMPT = (
    "a modern professional fashion model, full body portrait, "
    "wearing a modern black crop top and high waist skirt, "
    "realistic human proportions, natural skin texture, "
    "studio fashion photography, neutral studio background, "
    "high detail clothing, realistic fabric, photorealistic"
)

result = subprocess.run(
    [
        sys.executable,
        str(PROJECT / "generate_model.py"),
        "--prompt",      PROMPT,
        "--output_path", str(MODEL_IMAGE),
        "--steps",       "4",
        "--width",       "768",
        "--height",      "1024",
    ],
    capture_output=True,
    text=True,
    cwd=str(PROJECT),
)

print(result.stdout[-3000:] if result.stdout else "")
if result.returncode != 0:
    print(result.stderr[-5000:])
    raise RuntimeError("generate_model.py failed — see stderr above.")

assert MODEL_IMAGE.exists(), "Output image not created!"
print("✅ Generated:", MODEL_IMAGE)

In [ ]:
# ── STEP 7 — DISPLAY MODEL ─────────────────────────────────────────────────

from IPython.display import display
from PIL import Image

display(Image.open(MODEL_IMAGE).convert("RGB"))

In [ ]:
# ── STEP 8A — DISPLAY MODEL WITH COORDINATE GRID ──────────────────────────

import matplotlib.pyplot as plt
from PIL import Image

img = Image.open(MODEL_IMAGE).convert("RGB")
print("Image size:", img.size)

plt.figure(figsize=(9, 12))
plt.imshow(img)
plt.xlim(0, img.width)
plt.ylim(img.height, 0)
plt.xticks(range(0, img.width + 1, 100))
plt.yticks(range(0, img.height + 1, 100))
plt.grid(alpha=0.4)
plt.title("Click 3 points INSIDE the clothing region")
plt.show()

In [ ]:
# ── STEP 8B — ENTER THREE CLOTHING POINT COORDINATES ──────────────────────

points = []

for i in range(3):
    while True:
        raw = input(f"Point {i+1} (x,y): ").strip()
        try:
            x, y = [int(v.strip()) for v in raw.split(",")]
            if 0 <= x < img.width and 0 <= y < img.height:
                points.append([x, y])
                break
            print("❌ Point is outside the image.")
        except Exception:
            print("❌ Format must be: 380,420")

print("Selected points:", points)

In [ ]:
# ── STEP 8C — RUN SAM SEGMENTATION ────────────────────────────────────────

import gc
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
from segment_anything import SamPredictor, sam_model_registry

SAM_PATH  = str(PROJECT / "weights/sam_vit_h_4b8939.pth")
MASK_PATH = str(PROJECT / "reference_images/fashion_model_mask.png")

sam = sam_model_registry["vit_h"](checkpoint=SAM_PATH)
sam.to(device="cuda")
predictor = SamPredictor(sam)

rgb = np.array(Image.open(MODEL_IMAGE).convert("RGB"))
predictor.set_image(rgb)

input_point = np.array(points, dtype=np.float32)
input_label = np.ones(len(points), dtype=np.int32)

masks, scores, _ = predictor.predict(
    point_coords=input_point,
    point_labels=input_label,
    multimask_output=True,
)

best = int(np.argmax(scores))
mask = masks[best]

cv2.imwrite(MASK_PATH, (mask.astype(np.uint8) * 255))

print("Mask saved:", MASK_PATH)
print("SAM score:", float(scores[best]))

plt.figure(figsize=(9, 12))
plt.imshow(rgb)
plt.imshow(mask, alpha=0.45)
plt.scatter(input_point[:, 0], input_point[:, 1], c="red", s=80)
plt.axis("off")
plt.title("SAM Mask Preview")
plt.show()

del predictor, sam
gc.collect()
torch.cuda.empty_cache()
print("✅ SAM released from GPU.")

In [ ]:
# ── STEP 9 — UPLOAD GARMENT IMAGE ─────────────────────────────────────────

from google.colab import files
from shutil import copyfile
from PIL import Image
from IPython.display import display

uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No garment image uploaded.")

uploaded_name = next(iter(uploaded))
GARMENT_PATH  = str(PROJECT / "samples/garment.png")

copyfile(uploaded_name, GARMENT_PATH)

garment = Image.open(GARMENT_PATH).convert("RGB")
print("Garment size:", garment.size)
display(garment)
print("✅ Saved:", GARMENT_PATH)

In [ ]:
# ── STEP 10 — VERIFY IDM-VTON SRC ─────────────────────────────────────────

import os, sys
from pathlib import Path

os.chdir(PROJECT)

if str(IDM) not in sys.path:
    sys.path.insert(0, str(IDM))

assert (IDM / "src").exists(),         "idm_vton/src is missing"
assert Path(MODEL_IMAGE).exists(),     "Model image missing"
assert Path(MASK_PATH).exists(),       "Mask missing"
assert Path(GARMENT_PATH).exists(),    "Garment missing"
assert Path("try_on.py").exists(),     "try_on.py missing"

from src.tryon_pipeline import StableDiffusionXLInpaintPipeline
print("✅ IDM-VTON src import works.")

In [ ]:
# ── STEP 11 — RUN IDM-VTON VIRTUAL TRY-ON ─────────────────────────────────

import os, subprocess, sys
from pathlib import Path

CLOTH_TYPE  = "black crop top"   # ← change to match your garment
FINAL_IMAGE = str(PROJECT / "results/final_tryon.png")

env = os.environ.copy()
env["PYTHONPATH"] = str(IDM) + os.pathsep + env.get("PYTHONPATH", "")
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env["TOKENIZERS_PARALLELISM"]  = "false"

cmd = [
    sys.executable,
    str(PROJECT / "try_on.py"),
    "--reference_image", str(MODEL_IMAGE),
    "--mask",            MASK_PATH,
    "--garment",         GARMENT_PATH,
    "--cloth_type",      CLOTH_TYPE,
    "--output_path",     FINAL_IMAGE,
]

print("Running IDM-VTON...")
result = subprocess.run(cmd, cwd=str(PROJECT), env=env, capture_output=True, text=True)

print(result.stdout[-3000:] if result.stdout else "")
if result.returncode != 0:
    print(result.stderr[-5000:])
    raise RuntimeError("IDM-VTON failed — see stderr above.")

assert Path(FINAL_IMAGE).exists(), "Output image not created!"
print("✅ Final image:", FINAL_IMAGE)

In [ ]:
# ── STEP 12 — DISPLAY FINAL RESULT ────────────────────────────────────────

from IPython.display import display
from PIL import Image

result = Image.open(FINAL_IMAGE).convert("RGB")
print("FINAL RESULT:", result.size)
display(result)

# ✅ Complete

Expected output files:
```
reference_images/fashion_model.png       ← RealVisXL output
reference_images/fashion_model_mask.png  ← SAM mask
samples/garment.png                      ← uploaded garment
results/final_tryon.png                  ← IDM-VTON result
```